# UrbanSound8K — CNN Classification Pipeline
**Google Colab Setup Guide**

---

## Before you run anything — do these two steps once

### Step 1 — Upload your project to Google Drive
Upload the entire `thesis_code/` folder to your Google Drive.  
Recommended path: `My Drive/thesis_code/`

### Step 2 — Upload the dataset to Google Drive
Upload the `UrbanSound8K/` folder (6.6 GB) inside `thesis_code/data/`:  
`My Drive/thesis_code/data/UrbanSound8K/`

The expected structure in Drive:
```
My Drive/
└── thesis_code/
    ├── dataset/
    ├── models/
    ├── preprocessing/
    ├── train/
    ├── evaluate/
    ├── results/
    ├── saved_models/
    ├── main.py
    ├── requirements.txt
    └── data/
        └── UrbanSound8K/
            ├── audio/
            │   ├── fold1/ … fold10/
            └── metadata/
                └── UrbanSound8K.csv
```

---
**Runtime → Change runtime type → T4 GPU** before running.

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

## Cell 2 — Set project root and verify structure

In [ ]:
import os, sys

PROJECT_ROOT = '/content/drive/MyDrive/thesis_code'
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

required = [
    'main.py',
    'requirements.txt',
    'dataset/urbansound_dataset.py',
    'preprocessing/mel_spectrogram.py',
    'models/cnn.py',
    'train/train.py',
    'evaluate/evaluate.py',
    'data/UrbanSound8K/metadata/UrbanSound8K.csv',
    'data/UrbanSound8K/audio/fold1',
]

all_ok = True
for path in required:
    exists = os.path.exists(path)
    status = '✓' if exists else '✗  MISSING'
    print(f'  {status}  {path}')
    if not exists:
        all_ok = False

print()
if all_ok:
    print('All files found — ready to proceed.')
else:
    print('Fix the missing paths above before continuing.')

## Cell 3 — Install dependencies

In [ ]:
# Colab already has torch and torchaudio.
# We only need to add the packages that are missing.
!pip install -q librosa pandas scikit-learn torchattacks ptflops
print('Dependencies installed.')

## Cell 4 — Verify GPU and library versions

In [ ]:
import torch, torchaudio, librosa, sklearn

print(f'PyTorch      : {torch.__version__}')
print(f'torchaudio   : {torchaudio.__version__}')
print(f'librosa      : {librosa.__version__}')
print(f'scikit-learn : {sklearn.__version__}')
print()

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU : {gpu}  ({mem:.1f} GB)')
else:
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU.')

## Cell 5 — Smoke-test: preprocessing on one sample

In [ ]:
import sys, os
sys.path.insert(0, PROJECT_ROOT)

from preprocessing.mel_spectrogram import MelSpectrogramTransform, SAMPLE_RATE, N_MELS, N_TIME_FRAMES
from dataset.urbansound_dataset import UrbanSoundDataset

DATA_ROOT = os.path.join(PROJECT_ROOT, 'data', 'UrbanSound8K')

ds = UrbanSoundDataset(root_dir=DATA_ROOT, folds=[1])
spec, label = ds[0]

print(f'Dataset size   : {len(ds)} samples (fold 1)')
print(f'Spectrogram    : {tuple(spec.shape)}  (expect (1, {N_MELS}, {N_TIME_FRAMES}))')
print(f'Value range    : [{spec.min():.4f}, {spec.max():.4f}]  (expect [0, 1])')
print(f'Label          : {label}  → {ds.get_class_name(label)}')
assert spec.shape == (1, N_MELS, N_TIME_FRAMES)
assert 0.0 <= spec.min() and spec.max() <= 1.0
print('Preprocessing smoke-test passed.')

## Cell 6 — Smoke-test: CNN forward pass

In [ ]:
import torch
from models.cnn import UrbanSoundCNN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = UrbanSoundCNN().to(device)
dummy  = torch.zeros(8, 1, 64, 128, device=device)
logits = model(dummy)

total_params = sum(p.numel() for p in model.parameters())
print(f'Device         : {device}')
print(f'Input  shape   : {tuple(dummy.shape)}')
print(f'Output shape   : {tuple(logits.shape)}  (expect (8, 10))')
print(f'Parameters     : {total_params:,}')
assert logits.shape == (8, 10)
print('CNN forward pass smoke-test passed.')

## Cell 7 — Quick run: 100 samples per fold (sanity check)
Runs the full pipeline on a tiny subset to verify everything works end-to-end  
before committing to the full 50-epoch training run.

Expected time: ~5 minutes on T4 GPU.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, 'main.py', '--epochs', '3', '--quick', '100', '--num-workers', '2'],
    cwd=PROJECT_ROOT,
    capture_output=False,   # print output live
    text=True,
)
print('\nReturn code:', result.returncode)

## Cell 8 — Full 10-fold cross-validation training
Trains on all 8,732 samples for 50 epochs per fold.  

| Hardware | Time per fold | Total (10 folds) |
|---|---|---|
| T4 GPU (Colab free) | ~15–20 min | ~3–4 hours |
| A100 GPU (Colab Pro) | ~5–8 min | ~1 hour |
| CPU only | ~3–5 hours | ~30–50 hours |

> **Colab disconnects after ~90 min of inactivity.**  
> Each fold saves its checkpoint immediately after training, so if Colab  
> disconnects mid-run you only lose the current fold — not previous ones.

**Run this cell and keep the tab active (scroll / interact occasionally).**

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, 'main.py', '--epochs', '50', '--num-workers', '2'],
    cwd=PROJECT_ROOT,
    capture_output=False,
    text=True,
)
print('\nReturn code:', result.returncode)

## Cell 9 — View saved results

In [ ]:
import json, os, glob

results_dir = os.path.join(PROJECT_ROOT, 'results')
runs = sorted(os.listdir(results_dir))
latest_run = os.path.join(results_dir, runs[-1])
print(f'Latest run: {runs[-1]}')
print(f'Files:')
for f in sorted(os.listdir(latest_run)):
    size = os.path.getsize(os.path.join(latest_run, f))
    print(f'  {f:<30}  {size:>6} bytes')

In [ ]:
# Print the cross-validation summary
summary_path = os.path.join(latest_run, 'cv_summary.json')
with open(summary_path) as f:
    summary = json.load(f)

print('Per-fold results:')
print(f"{'Fold':>6}  {'Accuracy':>9}  {'Precision(W)':>13}  {'Recall(W)':>10}  {'F1(W)':>7}")
print('-' * 52)
for r in summary['per_fold']:
    print(f"  {r['fold']:>4}  {r['accuracy']*100:>8.2f}%  {r['precision_weighted']*100:>12.2f}%  {r['recall_weighted']*100:>9.2f}%  {r['f1_weighted']*100:>6.2f}%")

m = summary['mean']
s = summary['std']
print('=' * 52)
print(f"  Mean accuracy  : {m['accuracy']*100:.2f}%  ±  {s['accuracy']*100:.2f}%")
print(f"  Mean F1 (wtd)  : {m['f1_weighted']*100:.2f}%  ±  {s['f1_weighted']*100:.2f}%")

## Cell 10 — Plot training history

In [ ]:
import json, os, glob, matplotlib.pyplot as plt

fold_files = sorted(glob.glob(os.path.join(latest_run, 'fold_*_results.json')))

fig, axes = plt.subplots(2, 5, figsize=(20, 7))
axes = axes.flatten()

for i, fpath in enumerate(fold_files):
    with open(fpath) as f:
        data = json.load(f)
    h = data['training_history']
    ax = axes[i]
    ax.plot(h['train_acc'], label='Train acc', color='steelblue')
    ax.plot(h['val_acc'],   label='Val acc',   color='darkorange')
    ax.set_title(f"Fold {data['fold']}  (best val {data['best_val_acc']:.1f}%)")
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy (%)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Training vs Validation Accuracy — All 10 Folds', fontsize=14, fontweight='bold')
plt.tight_layout()
plot_path = os.path.join(latest_run, 'training_curves.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved to {plot_path}')

## Cell 11 — Evaluate a single fold checkpoint (optional)
Useful if you want to re-evaluate after the training run without retraining.

In [ ]:
import sys
sys.path.insert(0, PROJECT_ROOT)

from evaluate.evaluate import evaluate_checkpoint

FOLD      = 1     # change to any fold 1–10
DATA_ROOT = os.path.join(PROJECT_ROOT, 'data', 'UrbanSound8K')
CKPT      = os.path.join(PROJECT_ROOT, 'saved_models', f'best_fold{FOLD}.pt')

metrics = evaluate_checkpoint(
    checkpoint_path = CKPT,
    data_root       = DATA_ROOT,
    fold            = FOLD,
    num_workers     = 2,
    verbose         = True,
)

---
## Tips for working in Colab

| Problem | Solution |
|---|---|
| Colab disconnects | Checkpoints are saved after each fold — just re-run from the disconnected fold |
| Out of GPU memory | Reduce `--batch-size` to 16 |
| Slow data loading | Keep `--num-workers 2` (Colab has 2 CPU cores) |
| Want faster iteration | Run `--quick 100` first, then full run |
| Save results permanently | They are already in Google Drive under `thesis_code/results/` |